In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


try:
    base_dir = os.getcwd()
except NameError:
    base_dir = os.path.dirname(os.path.abspath(__file__))
sys.path.append(os.path.dirname(base_dir))

In [3]:
pred_path = "../PyKeen/results/lp/Prime_KGs/adni/ecdf/ComplEx/predictions.csv"
prediction = pd.read_csv(pred_path, index_col=0)
prediction

,head_id,head_label,relation_id,relation_label,tail_id,tail_label,score,true_labels
0,0,002_S_0413,37,reg_disease,705,APOD,-7.093873,0
1,0,002_S_0413,37,reg_disease,763,ARSG,-3.008037,0
2,0,002_S_0413,37,reg_disease,960,BORA,-5.349096,0
3,0,002_S_0413,37,reg_disease,1276,CILP,5.247251,0
4,0,002_S_0413,37,reg_disease,1407,CS,1.201004,0
...,...,...,...,...,...,...,...,...
14747,452,941_S_4365,38,reg_healthy,8328,"p(UniProtKB:""PLTP_HUMAN"")",-3.787958,0
14748,452,941_S_4365,38,reg_healthy,8369,"p(UniProtKB:""PTPRA_RAT"")",-2.867919,0
14749,452,941_S_4365,38,reg_healthy,8387,"p(UniProtKB:""RELN_HUMAN"")",-4.229211,0
14750,452,941_S_4365,38,reg_healthy,8393,"p(UniProtKB:""RHOA_MOUSE"")",-5.994485,0


In [11]:
prediction[(prediction['head_label']=='002_S_0413') & (prediction['relation_label']=='reg_healthy')]

,head_id,head_label,relation_id,relation_label,tail_id,tail_label,score,true_labels
19,0,002_S_0413,38,reg_healthy,8298,"p(UniProtKB:""PAX6_HUMAN"")",0.712296,0
20,0,002_S_0413,38,reg_healthy,8520,"p(UniProtKB:""UCN3_MOUSE"")",-0.792412,0


In [19]:
sample_label_map = prediction.set_index("head_label")['true_labels'].to_dict()
sample_label_map

{'002_S_0413': 0,
 '002_S_0685': 0,
 '002_S_4213': 0,
 '002_S_4521': 1,
 '003_S_1122': 1,
 '003_S_4119': 0,
 '003_S_4288': 0,
 '003_S_4350': 0,
 '003_S_4373': 1,
 '005_S_0448': 1,
 '005_S_0553': 0,
 '005_S_0572': 1,
 '005_S_4707': 1,
 '006_S_4346': 1,
 '006_S_4515': 1,
 '006_S_4546': 1,
 '007_S_4568': 1,
 '009_S_0842': 0,
 '009_S_4530': 1,
 '010_S_0161': 1,
 '010_S_4345': 0,
 '011_S_0023': 0,
 '011_S_4366': 1,
 '012_S_0637': 0,
 '012_S_4026': 0,
 '014_S_0548': 0,
 '014_S_0658': 1,
 '014_S_2185': 1,
 '014_S_4080': 0,
 '014_S_4401': 0,
 '016_S_0359': 0,
 '016_S_0702': 1,
 '016_S_4584': 1,
 '018_S_2155': 1,
 '018_S_4257': 0,
 '018_S_4313': 0,
 '018_S_4399': 0,
 '018_S_4696': 1,
 '019_S_4252': 1,
 '020_S_1288': 0,
 '021_S_0159': 0,
 '021_S_0984': 0,
 '021_S_4276': 0,
 '022_S_2087': 1,
 '022_S_4291': 0,
 '023_S_0061': 0,
 '023_S_0217': 1,
 '023_S_4020': 0,
 '023_S_4501': 1,
 '027_S_0120': 0,
 '027_S_0256': 1,
 '027_S_0408': 1,
 '029_S_4384': 0,
 '029_S_4385': 0,
 '031_S_0618': 0,
 '031_S_20

In [ ]:

i = 0
k = 2
def rank_predictions(prediction, T=1):
    
    patient_ids = prediction['head_label'].unique()
    sample_label_map = prediction.set_index("head_label")['true_labels'].to_dict()
    
    for patient_id in patient_ids:
        df_disease = prediction[(prediction['head_label'] == patient_id) & (prediction['relation_label'] == 'reg_disease')].copy()
        df_healthy = prediction[(prediction['head_label'] == patient_id) & (prediction['relation_label'] == 'reg_healthy')].copy()
        
        # 2. Calculate statistics
        mean_d, std_d = df_disease['score'].mean(), df_disease['score'].std()
        mean_h, std_h = df_healthy['score'].mean(), df_healthy['score'].std()
        
        # Handle edge case: if std is 0 or NaN (e.g., only 0 or 1 protein found), set std to 1 to avoid division by zero
        std_d = 1.0 if pd.isna(std_d) or std_d == 0 else std_d
        std_h = 1.0 if pd.isna(std_h) or std_h == 0 else std_h

        # 3. Assign Z-scores directly back to the dataframes
        df_disease['zscore'] = (df_disease['score'] - mean_d) / std_d
        df_healthy['zscore'] = (df_healthy['score'] - mean_h) / std_h
        df_sample = pd.concat([df_disease, df_healthy], axis=0).reset_index(drop=True)
        
        # 4. softmax apply to z-scores to get probabilities
        zscores = df_sample['zscore'].to_numpy()
        shifted_zscores = zscores - np.max(zscores)
        # Apply the Softmax formula with Temperature
        exp_scores = np.exp(shifted_zscores / T)
        softmax_probabilities = exp_scores / np.sum(exp_scores)
        df_sample['softmax_score'] = softmax_probabilities

        df_sorted = df_sample.sort_values(by='softmax_score', ascending=False).reset_index(drop=True)
        
        print(f"Number of disease proteins: {len(df_disease)}, mean score: {mean_d}")
        print(f"Number of healthy proteins: {len(df_healthy)}, mean score: {mean_h}")
        print(f"true label: {sample_label_map[patient_id]}")
        print("-" * 40)
        print(df_sorted[['tail_label', 'relation_label', 'score','zscore']])
    

Number of disease proteins: 154, mean score: -3.8416469121103898
Number of healthy proteins: 35, mean score: -4.493033039857143
true label: 0
----------------------------------------
                     tail_label relation_label      score    zscore
0      p(UniProtKB:"CLU_MOUSE")    reg_healthy   3.642567  1.867173
1                          TANK    reg_disease   2.045963  1.582059
2                           HTT    reg_disease   1.566048  1.453101
3                           CCK    reg_disease   1.559750  1.451409
4    p(UniProtKB:"FOXO4_HUMAN")    reg_healthy   1.698746  1.421054
..                          ...            ...        ...       ...
184                        MTRR    reg_disease -14.915038 -2.975530
185  p(UniProtKB:"ROCK2_MOUSE")    reg_healthy -18.402122 -3.192227
186                       PPARG    reg_disease -15.736403 -3.196239
187                       PRKCI    reg_disease -15.905630 -3.241712
188                       PTPRS    reg_disease -16.575407 -3.421688



In [31]:
patient_ids = prediction['head_label'].unique()
sample_label_map = prediction.set_index("head_label")['true_labels'].to_dict()
T = 1
for patient_id in patient_ids:
    df_disease = prediction[(prediction['head_label'] == patient_id) & (prediction['relation_label'] == 'reg_disease')].copy()
    df_healthy = prediction[(prediction['head_label'] == patient_id) & (prediction['relation_label'] == 'reg_healthy')].copy()
    
    # 2. Calculate statistics
    mean_d, std_d = df_disease['score'].mean(), df_disease['score'].std()
    mean_h, std_h = df_healthy['score'].mean(), df_healthy['score'].std()
    
    # Handle edge case: if std is 0 or NaN (e.g., only 0 or 1 protein found), set std to 1 to avoid division by zero
    std_d = 1.0 if pd.isna(std_d) or std_d == 0 else std_d
    std_h = 1.0 if pd.isna(std_h) or std_h == 0 else std_h

    # 3. Assign Z-scores directly back to the dataframes
    df_disease['zscore'] = (df_disease['score'] - mean_d) / std_d
    df_healthy['zscore'] = (df_healthy['score'] - mean_h) / std_h
    df_sample = pd.concat([df_disease, df_healthy], axis=0).reset_index(drop=True)
    
    # 4. softmax apply to z-scores to get probabilities
    zscores = df_sample['zscore'].to_numpy()
    shifted_zscores = zscores - np.max(zscores)
    # Apply the Softmax formula with Temperature
    exp_scores = np.exp(shifted_zscores / T)
    softmax_probabilities = exp_scores / np.sum(exp_scores)
    df_sample['softmax_score'] = softmax_probabilities

    df_sorted = df_sample.sort_values(by='softmax_score', ascending=False).reset_index(drop=True)
    
    print(f"Number of disease proteins: {len(df_disease)}, mean score: {mean_d}")
    print(f"Number of healthy proteins: {len(df_healthy)}, mean score: {mean_h}")
    print(f"true label: {sample_label_map[patient_id]}")
    print("-" * 40)
    print(df_sorted[['tail_label', 'relation_label','zscore','softmax_score']])
    break
    

Number of disease proteins: 19, mean score: -5.484108189473685
Number of healthy proteins: 2, mean score: -0.040058134999999995
true label: 0
----------------------------------------
                   tail_label relation_label    zscore  softmax_score
0                        CILP    reg_disease  1.871770       0.209654
1                       PRKCE    reg_disease  1.230765       0.110438
2                          CS    reg_disease  1.166021       0.103515
3   p(UniProtKB:"PAX6_HUMAN")    reg_healthy  0.707107       0.065418
4                         DCN    reg_disease  0.480204       0.052138
5                        ARSG    reg_disease  0.431878       0.049678
6                       GABRE    reg_disease  0.416714       0.048931
7                          TF    reg_disease  0.386128       0.047457
8                        INVS    reg_disease  0.319473       0.044397
9                        NEFM    reg_disease  0.280021       0.042679
10                        CTH    reg_disease  0

In [10]:
prediction[(prediction['head_label']=='002_S_0413') & (prediction['relation_label']=='reg_disease')]

,head_id,head_label,relation_id,relation_label,tail_id,tail_label,score,true_labels
0,0,002_S_0413,37,reg_disease,705,APOD,-7.093873,0
1,0,002_S_0413,37,reg_disease,763,ARSG,-3.008037,0
2,0,002_S_0413,37,reg_disease,960,BORA,-5.349096,0
3,0,002_S_0413,37,reg_disease,1276,CILP,5.247251,0
4,0,002_S_0413,37,reg_disease,1407,CS,1.201004,0
5,0,002_S_0413,37,reg_disease,1429,CSTB,-7.165911,0
6,0,002_S_0413,37,reg_disease,1438,CTH,-4.282099,0
7,0,002_S_0413,37,reg_disease,1533,DCN,-2.730972,0
8,0,002_S_0413,37,reg_disease,1611,DMD,-14.583633,0
9,0,002_S_0413,37,reg_disease,2057,GABRD,-8.311679,0


In [5]:
prediction['head_id'].unique().shape

(137,)

In [7]:
prediction['relation_label'].unique()

<StringArray>
['reg_disease', 'reg_healthy']
Length: 2, dtype: str

In [12]:
test_prediction = pd.read_csv("../PyKeen/results/lp/Prime_KGs/test_result/link_predictions.csv", index_col=0)
test_prediction

,head_id,head_label,relation_id,relation_label,tail_id,tail_label,score,true_labels
0,0,002_S_0413,38,reg_healthy,456,AAMP,-19.100306,0
1,0,002_S_0413,38,reg_healthy,458,AATF,-23.603207,0
2,0,002_S_0413,38,reg_healthy,459,ABAT,-16.800436,0
3,0,002_S_0413,38,reg_healthy,482,ACACA,-12.660259,0
4,0,002_S_0413,38,reg_healthy,485,ACADL,-18.646162,0
...,...,...,...,...,...,...,...,...
139187,451,941_S_4292,38,reg_healthy,8527,"p(UniProtKB:""VAMP3_HUMAN"")",-11.282839,0
139188,451,941_S_4292,38,reg_healthy,8530,"p(UniProtKB:""VAMP8_HUMAN"")",-16.731258,0
139189,451,941_S_4292,38,reg_healthy,8537,"p(UniProtKB:""VIP_HUMAN"")",-18.566980,0
139190,451,941_S_4292,38,reg_healthy,8541,"p(UniProtKB:""VLDLR_MOUSE"")",-12.849744,0


In [ ]:
test_prediction['head_label']